## Objetivo de la Notebook 3

En esta notebook se desarrollará la etapa de modelado para el problema de análisis de sentimiento. Para ello, se utilizará el conjunto de datos obtenido en la etapa de preprocesamiento, el cual contiene los tweets limpios y preparados para su utilización en algoritmos de aprendizaje automático.

En primer lugar, se realizará la carga y validación del dataset procesado, verificando su estructura y la distribución de las clases. Posteriormente, se prepararán las variables predictoras y la variable objetivo, realizando las transformaciones necesarias para su utilización en modelos de clasificación binaria.

A continuación, los textos serán convertidos a una representación numérica mediante la técnica **TF-IDF (Term Frequency - Inverse Document Frequency)**, permitiendo transformar el contenido textual en vectores que puedan ser interpretados por los algoritmos de Machine Learning.

Finalmente, se entrenarán y evaluarán distintos modelos de clasificación, comparando su desempeño mediante métricas como *Accuracy*, *Precision*, *Recall* y *F1-score*. Los resultados obtenidos permitirán seleccionar el modelo con mejor rendimiento para ser utilizado en las etapas posteriores del proyecto y compararlo posteriormente con modelos preentrenados, de acuerdo con los objetivos planteados en la consigna.


In [2]:
#Imports

import pandas as pd

from sklearn.model_selection import train_test_split

from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

Inspección inicial del dataset

In [3]:
df = pd.read_csv(
    "../data/processed/training_preprocessed.csv"
)

In [4]:
df.shape

(1593436, 2)

In [5]:
df.head()

,polarity,clean_text
0,0,be bummer shoulda get david carr third day
1,0,upset can not update facebook texte might cry ...
2,0,dive many time ball manage save 50 rest go bound
3,0,whole body feel itchy like fire
4,0,no be not behave be mad can not see


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1593436 entries, 0 to 1593435
Data columns (total 2 columns):
 #   Column      Non-Null Count    Dtype 
---  ------      --------------    ----- 
 0   polarity    1593436 non-null  int64 
 1   clean_text  1593436 non-null  object
dtypes: int64(1), object(1)
memory usage: 24.3+ MB


In [7]:
df["polarity"].value_counts()

polarity
0    796835
4    796601
Name: count, dtype: int64

In [8]:
#Conversion de polaridad a 0's y 1's para una mejor interpretación

df["polarity"] = df["polarity"].map({
    0: 0,
    4: 1
})

In [9]:
#Verificacion del cambio de polaridad

df["polarity"].value_counts()

polarity
0    796835
1    796601
Name: count, dtype: int64

In [10]:
#Separacion de variables

X = df["clean_text"]

y = df["polarity"]

In [11]:
#Division train y test

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [12]:
#Validacion

print(X_train.shape)
print(X_test.shape)

print()

print(y_train.value_counts(normalize=True))

print()

print(y_test.value_counts(normalize=True))

(1274748,)
(318688,)

polarity
0    0.500074
1    0.499926
Name: proportion, dtype: float64

polarity
0    0.500072
1    0.499928
Name: proportion, dtype: float64


## Vectorización mediante TF-IDF

Los algoritmos de Machine Learning no pueden trabajar directamente con texto, por lo que es necesario transformar cada documento en una representación numérica. Para ello se utilizará la técnica **TF-IDF (Term Frequency - Inverse Document Frequency)**.

TF-IDF asigna un peso a cada término considerando dos aspectos: su frecuencia dentro de un documento (*Term Frequency*) y su frecuencia en el conjunto completo de documentos (*Inverse Document Frequency*). De esta manera, las palabras que caracterizan mejor el contenido de un tweet reciben un mayor peso, mientras que aquellas demasiado frecuentes en el corpus reducen su importancia.

Para la construcción de la matriz TF-IDF se adoptaron las siguientes decisiones:

* **max_features = 50000:** limita el tamaño del vocabulario a las 50.000 características más relevantes, permitiendo representar el texto de forma eficiente sin incrementar excesivamente el costo computacional.
* **min_df = 5:** conserva únicamente los términos presentes en al menos cinco documentos, eliminando palabras extremadamente poco frecuentes que suelen corresponder a errores tipográficos o términos con escasa capacidad de generalización.
* **max_df = 0.95:** descarta los términos presentes en más del 95 % de los documentos, ya que aportan poca información discriminativa para la clasificación.
* **ngram_range = (1, 2):** incorpora tanto unigramas (palabras individuales) como bigramas (pares consecutivos de palabras), permitiendo capturar expresiones relevantes para el análisis de sentimiento, como por ejemplo *"not good"* o *"very happy"*.

Como resultado de esta etapa se obtendrá una representación vectorial del conjunto de entrenamiento y del conjunto de validación, la cual será utilizada como entrada para los modelos de clasificación que se entrenarán en las siguientes secciones.


In [13]:
# Configuración del vectorizador TF-IDF

vectorizer = TfidfVectorizer(
    max_features=50000,
    min_df=5,
    max_df=0.95,
    ngram_range=(1, 2)
)

In [14]:
# Ajuste del vocabulario y transformación del conjunto de entrenamiento

X_train_tfidf = vectorizer.fit_transform(X_train)

In [15]:
# Transformación del conjunto de prueba

X_test_tfidf = vectorizer.transform(X_test)

In [16]:
#Validaciones

print("Matriz de entrenamiento:", X_train_tfidf.shape)
print("Matriz de prueba:", X_test_tfidf.shape)

Matriz de entrenamiento: (1274748, 50000)
Matriz de prueba: (318688, 50000)


In [17]:
print("Cantidad de términos del vocabulario:")
print(len(vectorizer.vocabulary_))

Cantidad de términos del vocabulario:
50000


In [18]:
print("Primeros 20 términos del vocabulario:")

list(vectorizer.vocabulary_.keys())[:20]

Primeros 20 términos del vocabulario:


['be',
 'not',
 'even',
 'halfway',
 'yet',
 'follow',
 'get',
 'chair',
 'fan',
 'hand',
 'instead',
 'be not',
 'not even',
 'follow get',
 'always',
 'spend',
 'much',
 'money',
 'target',
 'spend much']

Resultado de la vectorización

La aplicación de TF-IDF generó una matriz de entrenamiento de 1.274.748 × 50.000 y una matriz de prueba de 318.688 × 50.000. El vocabulario alcanzó el límite máximo establecido de 50.000 términos, confirmando que el corpus posee una gran variedad léxica.

Además, la inspección de los primeros términos del vocabulario permitió verificar que la lematización fue aplicada correctamente y que los bigramas configurados mediante ngram_range=(1,2) fueron incorporados exitosamente, observándose expresiones como not even, be not y spend much.

Estos resultados indican que el proceso de vectorización se realizó correctamente y que las matrices obtenidas están listas para ser utilizadas en el entrenamiento de los modelos de clasificación.

## Entrenamiento del modelo Multinomial Naive Bayes


In [19]:
# Entrenamiento del modelo

nb_model = MultinomialNB()

nb_model.fit(X_train_tfidf, y_train)

,alpha,1.0
,force_alpha,True
,fit_prior,True
,class_prior,None


In [20]:
# Predicciones sobre el conjunto de prueba

# Predicciones sobre entrenamiento
y_pred_train_nb = nb_model.predict(X_train_tfidf)

# Predicciones sobre prueba
y_pred_test_nb = nb_model.predict(X_test_tfidf)

In [21]:
# ==========================
# Métricas sobre entrenamiento
# ==========================

accuracy_train_nb = accuracy_score(y_train, y_pred_train_nb)
precision_train_nb = precision_score(y_train, y_pred_train_nb)
recall_train_nb = recall_score(y_train, y_pred_train_nb)
f1_train_nb = f1_score(y_train, y_pred_train_nb)

# ==========================
# Métricas sobre prueba
# ==========================

accuracy_test_nb = accuracy_score(y_test, y_pred_test_nb)
precision_test_nb = precision_score(y_test, y_pred_test_nb)
recall_test_nb = recall_score(y_test, y_pred_test_nb)
f1_test_nb = f1_score(y_test, y_pred_test_nb)

In [22]:
print("=== Métricas sobre TRAIN ===")
print(f"Accuracy : {accuracy_train_nb:.4f}")
print(f"Precision: {precision_train_nb:.4f}")
print(f"Recall   : {recall_train_nb:.4f}")
print(f"F1-score : {f1_train_nb:.4f}")

print("\n=== Métricas sobre TEST ===")
print(f"Accuracy : {accuracy_test_nb:.4f}")
print(f"Precision: {precision_test_nb:.4f}")
print(f"Recall   : {recall_test_nb:.4f}")
print(f"F1-score : {f1_test_nb:.4f}")

=== Métricas sobre TRAIN ===
Accuracy : 0.7931
Precision: 0.7957
Recall   : 0.7886
F1-score : 0.7921

=== Métricas sobre TEST ===
Accuracy : 0.7821
Precision: 0.7844
Recall   : 0.7778
F1-score : 0.7811


In [23]:
print("=== Classification Report - TRAIN ===")
print(classification_report(y_train, y_pred_train_nb))

print("\n=== Classification Report - TEST ===")
print(classification_report(y_test, y_pred_test_nb))

=== Classification Report - TRAIN ===
              precision    recall  f1-score   support

           0       0.79      0.80      0.79    637468
           1       0.80      0.79      0.79    637280

    accuracy                           0.79   1274748
   macro avg       0.79      0.79      0.79   1274748
weighted avg       0.79      0.79      0.79   1274748


=== Classification Report - TEST ===
              precision    recall  f1-score   support

           0       0.78      0.79      0.78    159367
           1       0.78      0.78      0.78    159321

    accuracy                           0.78    318688
   macro avg       0.78      0.78      0.78    318688
weighted avg       0.78      0.78      0.78    318688



In [24]:
# Matriz de confusión sobre entrenamiento
cm_train_nb = confusion_matrix(y_train, y_pred_train_nb)

print("=== Matriz de Confusión - TRAIN ===")
print(cm_train_nb)

# Matriz de confusión sobre prueba
cm_test_nb = confusion_matrix(y_test, y_pred_test_nb)

print("\n=== Matriz de Confusión - TEST ===")
print(cm_test_nb)

=== Matriz de Confusión - TRAIN ===
[[508462 129006]
 [134748 502532]]

=== Matriz de Confusión - TEST ===
[[125316  34051]
 [ 35402 123919]]


Análisis de resultados

El modelo Multinomial Naive Bayes obtuvo un Accuracy de 79,31 % sobre el conjunto de entrenamiento y 78,21 % sobre el conjunto de prueba. De igual forma, las métricas de Precision, Recall y F1-score presentan valores muy similares entre ambos conjuntos, con una diferencia cercana al 1 %, lo que indica que el modelo mantiene un desempeño consistente al enfrentarse a datos no utilizados durante el entrenamiento.

El Classification Report muestra que ambas clases (sentimientos negativos y positivos) alcanzan métricas prácticamente idénticas, evidenciando un comportamiento equilibrado del clasificador y la ausencia de un sesgo significativo hacia alguna de las categorías. Esto también es esperable dado que el conjunto de datos utilizado para el entrenamiento se encuentra prácticamente balanceado.

Las matrices de confusión reflejan una distribución homogénea de los aciertos y errores en ambos conjuntos. Tanto en entrenamiento como en prueba, el modelo clasifica correctamente una proporción similar de ejemplos positivos y negativos, mientras que la cantidad de falsos positivos y falsos negativos se mantiene equilibrada.

La reducida diferencia entre las métricas obtenidas sobre train y test sugiere que el modelo no presenta evidencias significativas de overfitting, ya que mantiene un nivel de desempeño muy similar sobre datos vistos y no vistos durante el entrenamiento. En consecuencia, Multinomial Naive Bayes constituye una buena línea base para el problema de clasificación de sentimientos y servirá como referencia para comparar el rendimiento de modelos más complejos, como la Regresión Logística.


## Entrenamiento del modelo de Regresión Logística

Como segundo modelo de clasificación se utilizará **Regresión Logística**, uno de los algoritmos más utilizados en problemas de clasificación binaria y una referencia habitual en tareas de clasificación de texto.

Aunque su nombre hace referencia a una regresión, este algoritmo estima la probabilidad de pertenencia a una clase mediante una función logística, permitiendo clasificar cada documento como positivo o negativo.

In [25]:
# Creación y entrenamiento del modelo

lr_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

lr_model.fit(X_train_tfidf, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,42
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [26]:
# Predicciones sobre entrenamiento
y_pred_train_lr = lr_model.predict(X_train_tfidf)

# Predicciones sobre prueba
y_pred_test_lr = lr_model.predict(X_test_tfidf)

In [27]:
# ==========================
# Métricas sobre entrenamiento
# ==========================

accuracy_train_lr = accuracy_score(y_train, y_pred_train_lr)
precision_train_lr = precision_score(y_train, y_pred_train_lr)
recall_train_lr = recall_score(y_train, y_pred_train_lr)
f1_train_lr = f1_score(y_train, y_pred_train_lr)

# ==========================
# Métricas sobre prueba
# ==========================

accuracy_test_lr = accuracy_score(y_test, y_pred_test_lr)
precision_test_lr = precision_score(y_test, y_pred_test_lr)
recall_test_lr = recall_score(y_test, y_pred_test_lr)
f1_test_lr = f1_score(y_test, y_pred_test_lr)

In [28]:
print("=== Métricas sobre TRAIN ===")
print(f"Accuracy : {accuracy_train_lr:.4f}")
print(f"Precision: {precision_train_lr:.4f}")
print(f"Recall   : {recall_train_lr:.4f}")
print(f"F1-score : {f1_train_lr:.4f}")

print("\n=== Métricas sobre TEST ===")
print(f"Accuracy : {accuracy_test_lr:.4f}")
print(f"Precision: {precision_test_lr:.4f}")
print(f"Recall   : {recall_test_lr:.4f}")
print(f"F1-score : {f1_test_lr:.4f}")

=== Métricas sobre TRAIN ===
Accuracy : 0.8179
Precision: 0.8101
Recall   : 0.8304
F1-score : 0.8202

=== Métricas sobre TEST ===
Accuracy : 0.8042
Precision: 0.7967
Recall   : 0.8168
F1-score : 0.8066


In [29]:
print("=== Classification Report - TRAIN ===")
print(classification_report(y_train, y_pred_train_lr))

print("\n=== Classification Report - TEST ===")
print(classification_report(y_test, y_pred_test_lr))

=== Classification Report - TRAIN ===
              precision    recall  f1-score   support

           0       0.83      0.81      0.82    637468
           1       0.81      0.83      0.82    637280

    accuracy                           0.82   1274748
   macro avg       0.82      0.82      0.82   1274748
weighted avg       0.82      0.82      0.82   1274748


=== Classification Report - TEST ===
              precision    recall  f1-score   support

           0       0.81      0.79      0.80    159367
           1       0.80      0.82      0.81    159321

    accuracy                           0.80    318688
   macro avg       0.80      0.80      0.80    318688
weighted avg       0.80      0.80      0.80    318688



In [30]:
# Matriz de confusión sobre entrenamiento
cm_train_lr = confusion_matrix(y_train, y_pred_train_lr)

print("=== Matriz de Confusión - TRAIN ===")
print(cm_train_lr)

# Matriz de confusión sobre prueba
cm_test_lr = confusion_matrix(y_test, y_pred_test_lr)

print("\n=== Matriz de Confusión - TEST ===")
print(cm_test_lr)

=== Matriz de Confusión - TRAIN ===
[[513437 124031]
 [108051 529229]]

=== Matriz de Confusión - TEST ===
[[126163  33204]
 [ 29187 130134]]


### Análisis de resultados

La Regresión Logística obtuvo un desempeño superior al observado con el modelo Multinomial Naive Bayes, alcanzando un Accuracy del 81,79 % sobre el conjunto de entrenamiento y del 80,42 % sobre el conjunto de prueba. Asimismo, las métricas de Precision, Recall y F1-score presentan valores muy similares entre ambos conjuntos, con una diferencia aproximada del 1,4 %, lo que evidencia un comportamiento estable del modelo sobre datos no utilizados durante el entrenamiento.

El Classification Report muestra un rendimiento equilibrado para ambas clases, obteniendo valores de Precision, Recall y F1-score cercanos al 80 % tanto para los sentimientos negativos como positivos. En particular, el modelo logra un Recall ligeramente superior para la clase positiva, lo que indica una mejor capacidad para identificar correctamente los tweets con sentimiento positivo sin generar un incremento significativo de errores en la clase negativa.

Las matrices de confusión muestran una reducción en la cantidad de errores de clasificación respecto al modelo Multinomial Naive Bayes, especialmente en la disminución de los falsos negativos de la clase positiva. Esta mejora se refleja directamente en el incremento del Recall y del F1-score obtenidos por la Regresión Logística.

La reducida diferencia entre las métricas obtenidas sobre los conjuntos de train y test indica que el modelo no presenta evidencias significativas de overfitting, ya que mantiene un rendimiento muy similar sobre datos utilizados durante el entrenamiento y sobre datos no vistos previamente. En conjunto, estos resultados muestran que la Regresión Logística aprovecha de manera más efectiva la representación TF-IDF del texto, constituyéndose como el mejor modelo clásico evaluado en este trabajo y estableciendo una sólida referencia para la comparación con modelos preentrenados en las siguientes etapas del proyecto.


In [31]:
#Guardado de artefactos

import joblib

joblib.dump(lr_model, "../models/logistic_regression.joblib")
joblib.dump(nb_model, "../models/multinomial_nb.joblib")
joblib.dump(vectorizer, "../models/tfidf_vectorizer.joblib")

['../models/tfidf_vectorizer.joblib']